# KNN K 值選擇分析（臺北市房屋構造標準單價表）
用 CV RMSE + 一階/二階梯度找出最佳 K 值，產生三圖分析。

In [ ]:
# 1. Clone 專案（只需執行一次）
import os
if not os.path.exists('DataMiningExercises'):
    !git clone https://github.com/JackPan0521/DataMiningExercises.git
else:
    !git -C DataMiningExercises pull
print('Done')

In [ ]:
# 2. 安裝套件（Colab 通常已內建）
# !pip install scikit-learn pandas matplotlib numpy

In [ ]:
# 3. Imports
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold, cross_val_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [ ]:
# 4. 參數設定
K_MIN    = 1
K_MAX    = 20
CV_SPLITS = 5
CSV_PATH = 'DataMiningExercises/臺北市房屋構造標準單價表/臺北市房屋構造標準單價表-35層以下(112年7月起適用)-revised.csv'
OUT_CSV  = 'knn_k_selection_results.csv'
OUT_PNG  = 'knn_k_selection_gradient.png'

In [ ]:
# 5. 載入 CSV
last_error = None
for enc in ['utf-8-sig', 'cp950', 'big5', 'utf-8']:
    try:
        raw_df = pd.read_csv(CSV_PATH, encoding=enc)
        print(f'Loaded with encoding: {enc}, rows={len(raw_df)}')
        break
    except Exception as exc:
        last_error = exc
else:
    raise RuntimeError(f'Failed to read CSV; last error: {last_error}')
raw_df.head(3)

In [ ]:
# 6. 轉成長格式
def split_type(type_name):
    parts = str(type_name).split('-', 1)
    return (parts[0], parts[1]) if len(parts) == 2 else (parts[0], '一般')

value_cols = [c for c in raw_df.columns if c != '總層數']
long_df = raw_df.melt(id_vars=['總層數'], value_vars=value_cols, var_name='型別', value_name='單價')
parsed = long_df['型別'].apply(split_type)
long_df['構造'] = parsed.apply(lambda x: x[0])
long_df['類別'] = parsed.apply(lambda x: x[1])
long_df['單價'] = pd.to_numeric(long_df['單價'], errors='coerce')
train_df = long_df.dropna(subset=['單價']).copy()
print(f'Training rows: {len(train_df)}')
train_df.head(3)

In [ ]:
# 7. 建立 Pipeline
def build_pipeline(n_neighbors):
    preprocessor = ColumnTransformer(transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), ['總層數']),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), ['構造', '類別']),
    ])
    return Pipeline([('prep', preprocessor), ('model', KNeighborsRegressor(n_neighbors=n_neighbors, weights='distance'))])

In [ ]:
# 8. 搜尋各 K 的 CV RMSE
if K_MIN < 1 or K_MAX < K_MIN:
    raise ValueError('Invalid k range')

X = train_df[['總層數', '構造', '類別']]
y = train_df['單價']
cv = KFold(n_splits=CV_SPLITS, shuffle=True, random_state=42)

rows = []
for k in range(K_MIN, K_MAX + 1):
    scores = cross_val_score(build_pipeline(k), X, y, cv=cv, scoring='neg_mean_squared_error')
    rmse = np.sqrt(-scores)
    rows.append({'k': k, 'cv_rmse_mean': rmse.mean(), 'cv_rmse_std': rmse.std(ddof=1)})
    print(f'  k={k:2d}  CV RMSE={rmse.mean():.2f} ± {rmse.std(ddof=1):.2f}')

result = pd.DataFrame(rows)
result['gradient_1st'] = result['cv_rmse_mean'].diff()
result['gradient_2nd'] = result['gradient_1st'].diff()

In [ ]:
# 9. 選出最佳 K
best_k = int(result.loc[result['cv_rmse_mean'].idxmin(), 'k'])

finite_2nd = result['gradient_2nd'].replace([np.inf, -np.inf], np.nan).dropna()
elbow_k = int(result.loc[finite_2nd.abs().idxmax(), 'k']) if not finite_2nd.empty else best_k

print(f'Best k by minimum CV RMSE: {best_k}')
print(f'Elbow k by largest |2nd gradient|: {elbow_k}')

# Train RMSE
final_model = build_pipeline(best_k)
final_model.fit(X, y)
train_rmse = float(np.sqrt(mean_squared_error(y, final_model.predict(X))))
print(f'Train RMSE with best k={best_k}: {train_rmse:.2f}')

# K report
result.to_csv(OUT_CSV, index=False, encoding='utf-8-sig')
print(f'Saved: {OUT_CSV}')
result.sort_values('cv_rmse_mean').head(10)[['k','cv_rmse_mean','cv_rmse_std','gradient_1st','gradient_2nd']]

In [ ]:
# 10. 三圖視覺化
fig, axes = plt.subplots(3, 1, figsize=(10, 12), sharex=True)

axes[0].plot(result['k'], result['cv_rmse_mean'], marker='o', label='CV RMSE')
axes[0].fill_between(
    result['k'],
    result['cv_rmse_mean'] - result['cv_rmse_std'],
    result['cv_rmse_mean'] + result['cv_rmse_std'],
    alpha=0.2, label='±1 std'
)
axes[0].axvline(best_k,  color='red',   linestyle='--', label=f'Best k={best_k}')
axes[0].axvline(elbow_k, color='green', linestyle=':',  label=f'Elbow k={elbow_k}')
axes[0].set_ylabel('RMSE')
axes[0].set_title('K selection by CV RMSE')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(result['k'], result['gradient_1st'], marker='o', color='orange')
axes[1].axhline(0, color='black', linewidth=1)
axes[1].set_ylabel('1st Gradient')
axes[1].set_title('First gradient of RMSE')
axes[1].grid(True, alpha=0.3)

axes[2].plot(result['k'], result['gradient_2nd'], marker='o', color='purple')
axes[2].axhline(0, color='black', linewidth=1)
axes[2].set_ylabel('2nd Gradient')
axes[2].set_xlabel('k')
axes[2].set_title('Second gradient of RMSE')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 11. 儲存圖片並下載（選用）
fig.savefig(OUT_PNG, dpi=160)
print(f'Saved: {OUT_PNG}')

try:
    from google.colab import files
    files.download(OUT_CSV)
    files.download(OUT_PNG)
except ImportError:
    print('非 Colab 環境，略過下載步驟')